This notebook will pre-process the data and also carry out feature engineering

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, isnan, count, lit
from pyspark.sql.types import IntegerType, DoubleType, StringType, StructType, StructField

In [2]:
# Importing Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Create Spark session
spark = SparkSession.builder \
    .appName("Customer Churn Prediction") \
    .getOrCreate()

In [4]:
# Load Data
data_pp = spark.read.csv('../data/cleaned/cleaned_data.csv', header=True, inferSchema=True)

In [5]:
data_pp.show(5)

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|              No|         No|    

In [6]:
data_pp.select('MultipleLines').distinct().show()

+----------------+
|   MultipleLines|
+----------------+
|No phone service|
|              No|
|             Yes|
+----------------+



In [7]:
# Data Type Conversion

data_pp = data_pp.withColumn("SeniorCitizen", col("SeniorCitizen").cast(IntegerType()))
data_pp = data_pp.withColumn("tenure", col("tenure").cast(IntegerType()))
data_pp = data_pp.withColumn("MonthlyCharges", col("MonthlyCharges").cast(DoubleType()))
data_pp = data_pp.withColumn("TotalCharges", col("TotalCharges").cast(DoubleType()))

In [8]:
data_pp = data_pp.drop('CustomerID')

In [9]:
# Encoding Categorical Variables

from pyspark.ml.feature import StringIndexer, OneHotEncoder

multi_cat_cols = ["gender","MultipleLines","InternetService","Contract","PaymentMethod"]

# Indexing
indexers = [StringIndexer(inputCol=c, outputCol=c+"_index") for c in multi_cat_cols]

from pyspark.ml import Pipeline

pipeline = Pipeline(stages=indexers)
data_pp = pipeline.fit(data_pp).transform(data_pp)

# One-Hot Encoding
encoder = OneHotEncoder(inputCols=[c+"_index" for c in multi_cat_cols],
                        outputCols=[c+"_vec" for c in multi_cat_cols])
data_pp = encoder.fit(data_pp).transform(data_pp)

In [10]:
binary_cols = ["Partner","Dependents","PhoneService","OnlineSecurity","OnlineBackup",
               "DeviceProtection","TechSupport","StreamingTV","StreamingMovies","PaperlessBilling","Churn"]

for col_name in binary_cols:
    data_pp = data_pp.withColumn(col_name, 
                                 (col(col_name) == "Yes").cast(IntegerType()))

In [11]:
data_pp.show(5)

+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+------------+-------------------+---------------------+--------------+-------------------+-------------+-----------------+-------------------+-------------+-----------------+
|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|gender_index|MultipleLines_index|InternetService_index|Contract_index|PaymentMethod_index|   gender_vec|MultipleLines_vec|InternetService_vec| Contract_vec|PaymentMethod_vec|
+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+---------

In [12]:
from pyspark.ml.feature import VectorAssembler

feature_cols = ["SeniorCitizen","tenure","MonthlyCharges","TotalCharges"] + \
               [c+"_vec" for c in multi_cat_cols] + binary_cols[:-1]  # exclude Churn

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data_pp = assembler.transform(data_pp)

In [13]:
train_data, test_data = data_pp.randomSplit([0.8,0.2], seed=42)

In [14]:
# Feature Engineering (Tenure Grouping)

from pyspark.sql.functions import when

data_pp = data_pp.withColumn(
    "tenure_group",
    when(col("tenure") <= 12, "0-12")
    .when((col("tenure") > 12) & (col("tenure") <= 24), "12-24")
    .when((col("tenure") > 24) & (col("tenure") <= 48), "24-48")
    .otherwise("48+")
)

In [15]:
# Average Monthly Charges

data_pp = data_pp.withColumn(
    "AvgMonthlyCharges",
    col("TotalCharges") / col("tenure")
)

In [17]:
# Service Count Features

service_cols = ["PhoneService","MultipleLines","InternetService","OnlineSecurity",
                "OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]

from pyspark.sql.functions import sum as spark_sum

data_pp = data_pp.withColumn("TotalServices", sum([col(c) for c in service_cols]))


In [18]:
# Payment Behavior 

data_pp = data_pp.withColumn(
    "IsElectronicCheck",
    (col("PaymentMethod") == "Electronic check").cast("int")
)


In [19]:
# Contract Type

data_pp = data_pp.withColumn(
    "IsMonthToMonth",
    (col("Contract") == "Month-to-month").cast("int")
)
